In [3]:
import numpy as np
from itertools import product
from pathlib import Path

DATA_PATH = Path("../data/grids_dft_3_0.005")


def load_data(name, input_mat, output_mat, weights_mat):
    data = np.load(Path(f"{DATA_PATH}") / f"data_{name}.npz")

    input_ = data["rho_inv_4_norm"]
    output_ = data["exc_over_dm_cc_grids"]
    weights_ = data["weights"]

    for i_coord in range(len(output_)):
        input_mat.append(input_[:, i_coord])
        output_mat.append(output_[i_coord])
        weights_mat.append(weights_[i_coord])


input_mat = []
output_mat = []
weights_mat= []
basis = "cc-pVDZ"
molecular_list = [
    "methane",
    # "ethane",
    # "ethylene",
    # "acetylene",
]
extend_atom = ["0"]
extend_xyz = [1]
distance_list = [0]

for (
    name_mol,
    extend_atom,
    extend_xyz,
    distance,
) in product(
    molecular_list,
    extend_atom,
    extend_xyz,
    distance_list,
):
    name = f"{name_mol}_{basis}_{extend_atom}_{extend_xyz}_{distance:.4f}"

    if "openshell" in name:
        for i_spin in range(2):
            name_ = f"{name}_{i_spin}"
            if not (Path(f"{DATA_PATH}") / f"data_{name_}.npz").exists():
                print(f"No file: {name_}:>40")
                continue
            load_data(name_, input_mat, output_mat, weights_mat)
    else:
        if not (Path(f"{DATA_PATH}") / f"data_{name}.npz").exists():
            print(f"No file: {name:>40}")
            continue
        load_data(name, input_mat, output_mat, weights_mat)

In [4]:
print(np.shape(input_mat), np.shape(output_mat))

(48500, 4) (48500,)


In [5]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr = GridSearchCV(
    SVR(kernel="rbf", gamma=0.1),
    param_grid={
        "C": [1e0, 1e1, 1e2, 1e3],
        "gamma": np.logspace(-2, 2, 5),
    },
)

# random split
from sklearn.model_selection import train_test_split

input_mat = np.array(input_mat)
output_mat = np.array(output_mat)
weights_mat = np.array(weights_mat)
x_train, x_test, y_train, y_test, w_train, w_test = train_test_split(
    input_mat, output_mat, weights_mat, train_size=0.2
)

svr.fit(x_train, y_train, sample_weight=w_train)

ModuleNotFoundError: No module named 'sklearn'

In [21]:
import numpy as np
from itertools import product
from pathlib import Path

DATA_PATH = Path("../data/grids_dft")


def load_data(name, input_mat, output_mat, weights_mat):
    data = np.load(Path(f"{DATA_PATH}") / f"data_{name}.npz")

    input_ = data["rho_inv_4_norm"]
    output_ = data["exc_over_dm_cc_grids"]
    weights_ = data["weights"]

    for i_coord in range(len(output_)):
        input_mat.append(input_[:, i_coord])
        output_mat.append(output_[i_coord])
        weights_mat.append(weights_[i_coord])


input_mat = []
output_mat = []
weights_mat = []
basis = "cc-pVDZ"
molecular_list = ["methane"]
extend_atom = ["0", "0-1"]
extend_xyz = [1]
distance_list = [0]

for (
    name_mol,
    extend_atom,
    extend_xyz,
    distance,
) in product(
    molecular_list,
    extend_atom,
    extend_xyz,
    distance_list,
):
    name = f"{name_mol}_{basis}_{extend_atom}_{extend_xyz}_{distance:.4f}"

    if "openshell" in name:
        for i_spin in range(2):
            name_ = f"{name}_{i_spin}"
            if not (Path(f"{DATA_PATH}") / f"data_{name_}.npz").exists():
                print(f"No file: {name_}:>40")
                continue
            load_data(name_, input_mat, output_mat, weights_mat)
    else:
        if not (Path(f"{DATA_PATH}") / f"data_{name}.npz").exists():
            print(f"No file: {name:>40}")
            continue
        load_data(name, input_mat, output_mat, weights_mat)

input_mat = np.array(input_mat)
output_mat = np.array(output_mat)
weights_mat = np.array(weights_mat)

No file:             methane_cc-pVDZ_0-1_1_0.0000


In [23]:
output_mat_predict = svr.predict(input_mat)
print(
    np.sum(
        output_mat_predict * weights_mat * input_mat[:, 0]
        - output_mat * weights_mat * input_mat[:, 0]
    )
)
print(np.mean(output_mat_predict - output_mat))

0.6102213848685116
0.016612382387900276


In [6]:
import numpy as np
array = np.linspace(0, 8, 9)
print(array)
print(array[9 // 2 - 1 : 9 // 2 + 1])

[0. 1. 2. 3. 4. 5. 6. 7. 8.]
[3. 4.]


In [7]:
2**15

32768